In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
models ={
    'Omni (ZTF Single Channel)': ['../../models/BTSv2-pro/fanciful-gorge-557',
                               '../../models/BTSv2-pro/flowing-dream-569',
                               '../../models/BTSv2-pro/exalted-aardvark-570',
                               '../../models/BTSv2-pro/glamorous-thunder-571',
                               '../../models/BTSv2-pro/morning-feather-572'],
    'Omni (PS Triple Channel)':['../../models/BTSv2_PSonly/woven-sweep-9',
                                '../../models/BTSv2_PSonly/smooth-haze-141',
                                '../../models/BTSv2_PSonly/splendid-sun-142',
                                '../../models/BTSv2_PSonly/sweet-glade-143',
                                '../../models/BTSv2_PSonly/dazzling-thunder-144'
                                          ], 
}


In [3]:
days = 2**np.arange(0, 11)

In [4]:
def get_df(model_path, depth, day):


    if model_path in models['Image Backbone (ZTF Single Channel)'] or model_path in models['Image Backbone (PS Triple Channel)']:
        path = f"{model_path}/reports/depth{depth}/report_trigger+1.csv"
    else:
        path = f"{model_path}/reports/depth{depth}/report_trigger+{day}.csv"
    df = pd.read_csv(path)
    return df
        

In [5]:
def get_tables_for_all_models(model_choice, depth, day):

    tables = []
    cols = None
    names = None

    for model_path in models[model_choice]:
        df = get_df(model_path, depth, day)
        tables.append(df.apply(pd.to_numeric, errors='coerce').to_numpy()[:, 1:])
        cols = df.columns[1:]
        names = df['Class'].to_numpy()

    # find the mean and std for each metric across the 5 runs
    mean_table = np.mean(tables, axis=0)
    std_table = np.std(tables, axis=0)

    # add the columns and rows back from the first table
    new_df = pd.DataFrame()

    new_df['Class'] = names
    for i, col in enumerate(cols):

        # add the mean +/- std as a sring
        new_df[col] = [f"{mean_table[j, i]:.2f}±{std_table[j, i]:.2f}" for j in range(mean_table.shape[0])]

    return new_df
        
    

In [6]:
def get_f1_scores(model_choice, depth, days=[1,8,128]):

    tables = []

    for d in days:
        df = get_tables_for_all_models(model_choice, depth, d)
        
        # rename column to include the days like f1_d
        df.rename(columns={'f1-score': f'{model_choice} F1_{{{d}}}'}, inplace=True)
        df.drop(columns=['precision', 'recall', 'support'], inplace=True)

        tables.append(df)

    tables = [df.set_index('Class') for df in tables]

    # join the tables on the class
    return pd.concat(tables, axis=1)


In [7]:
def get_model_comparison_f1(depth, days=[1]):

    tables = []
    for model in models:

        if model=='Image Backbone':
            continue
    
        df = get_f1_scores(model, depth, days)
        tables.append(df)

    # join the tables on the class
    return pd.concat(tables, axis=1)

In [8]:
get_model_comparison_f1(2)

KeyError: 'Image Backbone (ZTF Single Channel)'

In [ ]:
print(get_model_comparison_f1(2).to_latex())

\begin{tabular}{lll}
\toprule
 & Image Backbone (ZTF Single Channel) F1_{1} & Image Backbone (PS Triple Channel) F1_{1} \\
Class &  &  \\
\midrule
AGN & 0.82±0.01 & 0.85±0.01 \\
CV & 0.70±0.01 & 0.80±0.02 \\
SLSN & 0.09±0.06 & 0.09±0.12 \\
SN-II & 0.36±0.02 & 0.41±0.04 \\
SN-Ia & 0.67±0.03 & 0.75±0.02 \\
SN-Ib/c & 0.11±0.02 & 0.09±0.01 \\
Varstar & 0.80±0.02 & 0.87±0.01 \\
accuracy & 0.64±0.02 & 0.71±0.01 \\
macro avg & 0.51±0.01 & 0.55±0.02 \\
weighted avg & 0.65±0.01 & 0.71±0.01 \\
\bottomrule
\end{tabular}



In [ ]:
print(get_model_comparison_f1(1).to_latex())

\begin{tabular}{lll}
\toprule
 & Image Backbone (ZTF Single Channel) F1_{1} & Image Backbone (PS Triple Channel) F1_{1} \\
Class &  &  \\
\midrule
Persistent & 0.86±0.00 & 0.88±0.01 \\
Transient & 0.92±0.00 & 0.93±0.00 \\
accuracy & 0.90±0.00 & 0.91±0.00 \\
macro avg & 0.89±0.00 & 0.91±0.00 \\
weighted avg & 0.90±0.00 & 0.91±0.00 \\
\bottomrule
\end{tabular}

